<a href="https://colab.research.google.com/github/Shukti-cyber/ML/blob/main/Elasticnet_Regression_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Elasticnet Regression

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df=pd.read_excel('insurance.xlsx')
df.head()

,age,sex,bmi,children,smoker,region,expenses
0,19,female,27.9,0,yes,southwest,16884.92
1,18,male,33.8,1,no,southeast,1725.55
2,28,male,33.0,3,no,southeast,4449.46
3,33,male,22.7,0,no,northwest,21984.47
4,32,male,28.9,0,no,northwest,3866.86


In [ ]:
df.shape

(1338, 7)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   expenses  1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


In [ ]:
categorical=[]
continuous=[]
check=[]
d_type=dict(df.dtypes)
for name,type in d_type.items():
  if str(type)=='object':
    categorical.append(name)
  elif str(type)=='float64':
    continuous.append(name)
  else:
    check.append(name)
print("categorical features :",categorical)
print("continuous features",continuous)
print("features to be checked ", check)

categorical features : ['sex', 'smoker', 'region']
continuous features ['bmi', 'expenses']
features to be checked  ['age', 'children']


In [ ]:
d_types = dict(df.dtypes)
for name,type_ in d_types.items():
  if str(type_)=='object':
      print(f"<=========={name}========>")
      print(df[name].value_counts())

<==========sex========>
sex
male      676
female    662
Name: count, dtype: int64
<==========smoker========>
smoker
no     1064
yes     274
Name: count, dtype: int64
<==========region========>
region
southeast    364
southwest    325
northwest    325
northeast    324
Name: count, dtype: int64


In [ ]:
df.describe()

,age,bmi,children,expenses
count,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,30.665471,1.094918,13270.422414
std,14.049960,6.098382,1.205493,12110.011240
min,18.000000,16.000000,0.000000,1121.870000
25%,27.000000,26.300000,0.000000,4740.287500
50%,39.000000,30.400000,1.000000,9382.030000
75%,51.000000,34.700000,2.000000,16639.915000
max,64.000000,53.100000,5.000000,63770.430000


In [ ]:
df.corr(numeric_only=True)

,age,bmi,children,expenses
age,1.000000,0.109341,0.042469,0.299008
bmi,0.109341,1.000000,0.012645,0.198576
children,0.042469,0.012645,1.000000,0.067998
expenses,0.299008,0.198576,0.067998,1.000000


In [ ]:
df.isnull().sum()

,0
age,0
sex,0
bmi,0
children,0
smoker,0
region,0
expenses,0


In [ ]:
df.drop('region',axis=1,inplace=True)

In [ ]:
df['sex'] = df['sex'].replace({'male': 0, 'female': 1})
df['smoker'] = df['smoker'].replace({'yes': 1, 'no': 0})

/tmp/ipykernel_2584/1097117560.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sex'] = df['sex'].replace({'male': 0, 'female': 1})
/tmp/ipykernel_2584/1097117560.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['smoker'] = df['smoker'].replace({'yes': 1, 'no': 0})


In [ ]:
X=df.drop('expenses',axis=1)
y=df['expenses']

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

Lasso Regression with default parameters

In [ ]:
#Modelling
from sklearn.linear_model import ElasticNet
enr_base=ElasticNet()
enr_base.fit(X_train,y_train)

#prediction
train_prediction = enr_base.predict(X_train)
test_prediction = enr_base.predict(X_test)

#evaluation
print("Train R2",enr_base.score(X_train,y_train))
print("Test R2",enr_base.score(X_test,y_test))
from sklearn.model_selection import cross_val_score
print("Cross Validation Score",cross_val_score(enr_base,X_train,y_train,cv=5).mean())


Train R2 0.3824976446106315
Test R2 0.4178740802692563
Cross Validation Score 0.37510435579770346


In [ ]:
from sklearn.model_selection import GridSearchCV

#model
estimator = ElasticNet()

#parameters & values (corrected "11_ratio" typo to "l1_ratio")
param_grid = {"alpha" : [0, 1, 2, 3, 5, 10], "l1_ratio": [0.1, 0.5, 0.75, 0.9, 0.95, 1]}

#Identifying the best value of the parameter within given values for the given data
model_hp = GridSearchCV(estimator,param_grid,cv=5)
model_hp.fit(X_train,y_train)
model_hp.best_params_

/usr/local/lib/python3.13/dist-packages/sklearn/base.py:1389: UserWarning: With alpha=0, this algorithm does not converge well. You are advised to use the LinearRegression estimator
  return fit_method(estimator, *args, **kwargs)
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: UserWarning: Coordinate descent with no regularization may lead to unexpected results and is discouraged.
  model = cd_fast.enet_coordinate_descent(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.573e+10, tolerance: 1.242e+07 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent

{'alpha': 10, 'l1_ratio': 1}

In [ ]:
# Train final ElasticNet model using the optimal hyperparameters found
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import cross_val_score

# Instantiate model with optimal parameters
enr_tuned = ElasticNet(alpha=10, l1_ratio=1)
enr_tuned.fit(X_train, y_train)

# Predict
train_pred_tuned = enr_tuned.predict(X_train)
test_pred_tuned = enr_tuned.predict(X_test)

# Evaluate performance
print("--- Tuned ElasticNet Model ({'alpha': 10, 'l1_ratio': 1}) ---")
print("Train R2:", enr_tuned.score(X_train, y_train))
print("Test R2:", enr_tuned.score(X_test, y_test))
print("Cross Validation Score:", cross_val_score(enr_tuned, X_train, y_train, cv=5).mean())

--- Tuned ElasticNet Model ({'alpha': 10, 'l1_ratio': 1}) ---
Train R2: 0.7411094487260133
Test R2: 0.7809614490519589
Cross Validation Score: 0.7339176249302982


In [ ]:
input_data = {"age0":31,"sex1":1,"bmi0":28,"children0":0,"smoker1":"no","region":"northeast"}

In [ ]:
# 1. Map and clean the keys to match the trained features
cleaned_input = {
    'age': input_data['age0'],
    'sex': input_data['sex1'], # Already mapped to 1
    'bmi': input_data['bmi0'],
    'children': input_data['children0'],
    'smoker': 1 if input_data['smoker1'] == 'yes' else 0
}

# 2. Convert to a DataFrame (1 row)
input_df = pd.DataFrame([cleaned_input])

# 3. Make prediction using the tuned model
predicted_expense = enr_tuned.predict(input_df)[0]

print(f"Preprocessed input features:\n{input_df}\n")
print(f"Predicted insurance expenses: ${predicted_expense:,.2f}")

Preprocessed input features:
   age  sex  bmi  children  smoker
0   31    1   28         0       0

Predicted insurance expenses: $5,042.86


In [ ]:
df_test = pd.DataFrame(input_data, index=[0])

# Rename columns to match the training features (removing the trailing numbers)
df_test.rename(columns={'age0': 'age', 'sex1': 'sex', 'bmi0': 'bmi', 'children0': 'children', 'smoker1': 'smoker'}, inplace=True)

# Drop unused region feature
df_test.drop('region', axis=1, errors='ignore', inplace=True)

# Safely replace values
df_test['sex'] = df_test['sex'].replace({'male': 0, 'female': 1})
df_test['smoker'] = df_test['smoker'].replace({'yes': 1, 'no': 0})

transform_data = df_test

/tmp/ipykernel_2584/4046124841.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_test['smoker'] = df_test['smoker'].replace({'yes': 1, 'no': 0})


In [ ]:
enr_tuned.predict(transform_data)

array([5042.85884112])